# Exercise 12: Cross validation

In this exercise, we'll practice implementing cross validation techniques, including leave-one-out and k-fold cross validation. We'll use the `PimaIndiansDiabetes2` practice dataset, which has medical data on a group of Pima Native American women, including whether or not they have diabetes. This dataset is part of the `mlbench` package. We'll be using each person's medical history to predict whether or not they have been diagnosed with diabetes.

---
## 1. Load Data (1 point)

Load the `tidyverse`, `boot`, and `mlbench` packages (you may need to install `boot` and `mlbench`).

Load the `PimaIndiansDiabetes2` dataset using the `data()` function. Drop the `insulin` column (it just has a lot of missing data) and then drop `NA`s from the rest of the dataset. Save your updated dataset to a new variable name. Finally, print the dimensions of your new dataset, and look at the first few lines of data.

In [42]:
# install.packages("boot", dependencies = TRUE)
# install.packages("mlbench", dependencies = TRUE)

library(tidyverse)
library(boot)
library(mlbench)

#load original dataframe
data("PimaIndiansDiabetes2")
my_data <- PimaIndiansDiabetes2

my_data$insulin <- NULL #remove the "insulin" column
cleaned_data <- drop_na(my_data) #drop NAs
head(cleaned_data,10)
dim(cleaned_data)

,pregnant,glucose,pressure,triceps,mass,pedigree,age,diabetes
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>
1,6,148,72,35,33.6,0.627,50,pos
2,1,85,66,29,26.6,0.351,31,neg
4,1,89,66,23,28.1,0.167,21,neg
5,0,137,40,35,43.1,2.288,33,pos
7,3,78,50,32,31.0,0.248,26,pos
9,2,197,70,45,30.5,0.158,53,pos
14,1,189,60,23,30.1,0.398,59,pos
15,5,166,72,19,25.8,0.587,51,pos
17,0,118,84,47,45.8,0.551,31,pos


[1] 532   8

(Note that in medical contexts, `pedigree` refers to a system of measuring family history of a condition. So here, higher numbers mean greater family history of diabetes. You can read more about this dataset [here](https://rdrr.io/cran/mlbench/man/PimaIndiansDiabetes.html).)

---
## 2. Leave-one-out Cross Validation (4 points)

In the tutorial, we learned how to fit leave-one-out cross validation using the `cv.glm` function from the `boot` package. But we can also do this manually using `predict()` like we have in the past.

Let's predict `diabetes`, a dichotomous outcome, using all the other variables in our modified dataset.

First, fit a logistic regression model using all of the observations except the very first one. Then use your fitted model to predict whether your holdout case is positive or negative for diabetes. Remember that in logistic regression, the model output (before applying the sigmoid function) is in **log-odds**. If this output is positive, the predicted probability is greater than 50%; if it is negative, the predicted probability is less than 50%.

Compare your prediction to the actual response for the first observation that you initially excluded. Did your model correctly classify this observation?

In [50]:
# head(cleaned_data,10)
# dim(cleaned_data)
train_set <- cleaned_data[2:nrow(cleaned_data), ] #create train set that includes all observations except the 1st observation
# head(train_set,10)
# dim(train_set)

test_set <-cleaned_data[1, ]
# head(test_set)

#use logistic regression to predict diabetes for above training set
model <- glm(diabetes ~ pregnant + glucose + pressure + triceps + mass + pedigree + age, 
             family = "binomial",
             data = train_set) 

holdout_pred <- predict(model, newdata = test_set)
print(holdout_pred)

        1 
0.9920213 


So we just calculated a single iteration of leave-one-out cross validation. We used 531 rows of our data to fit a model to predict the outcome of the last row.

Below, use a `for` loop to iterate through the rest of your dataset doing the same thing. You will need to:
* Create a data frame `results` with two columns: one named `actual` which holds the true classification for each observation, and one named `predicted`, which should be filled with `NA`s. This is where you'll store the output of your loop.
* Create a loop that runs through each row of your data, pulls that observation out, trains your model on the remaining data, and then tests the fitted model on your test observation.
* Store your model *predictions* ("pos" or "neg" -- not the log-odds) in the `predicted` column of your `results` dataframe

After you run your loop, print the first few lines of `results`.

In [71]:
# Initialize `results` data frame
results  <- data.frame(actual = cleaned_data$diabetes,
                     predicted = rep(NA, nrow(cleaned_data)))

#head(results)
#for loop
for (i in 1:nrow(cleaned_data)){ 
    # separate individual observation `i` from the rest of your data
    test_set <- cleaned_data[i, ] #the one row in loop 
    train_set <- cleaned_data[-i, ] #all the other rows
    
    # train your model
    model <- glm(diabetes ~ pregnant + glucose + pressure + triceps + mass + pedigree + age, 
             family = "binomial",
             data = train_set)
    
    # test model on hold out observation
    
    holdout_pred <- predict(model, newdata = test_set)
    # classify model prediction as "pos" or "neg" and add to `results`
    if (holdout_pred < 0) {
        pred = "neg" 
        results$predicted[i] <- pred 
    } else {
        pred = "pos" 
        results$predicted[i] <- pred 
    }
    rm(pred) #remove these variables just in case their values remain between loop iterations (unlikely)
    rm(test_set)
    rm(train_set)
    rm(model)
    }

head(results,10)

,actual,predicted
,<fct>,<chr>
1,pos,pos
2,neg,neg
3,neg,neg
4,pos,pos
5,pos,neg
6,pos,pos
7,pos,pos
8,pos,pos
9,pos,neg


Now, calculate the overall error of your model. What proportion of cases were incorrectly classified?

In [76]:
confusion_df <- data.frame(predicted = results$predicted,actual = results$actual)
table(confusion_df)
print("---")
print(paste("Accuracy:",mean(confusion_df$predicted == confusion_df$actual)))
error = 1 - mean(confusion_df$predicted == confusion_df$actual)
print(error)

         actual
predicted neg pos
      neg 315  78
      pos  40  99

[1] "---"
[1] "Accuracy: 0.778195488721805"
[1] 0.2218045


----
## 3. Compare to `cv.glm` (3 points)

Now, let's compare this result to the `cv.glm` function. Using the tutorial as a guide, use `cv.glm` to run LOOCV on the data, using the same model (i.e. still using all of the variables to predict diabetes diagnosis).

Note that, because this is a `classification` problem and not a regression problem like in the tutorial, we need to adjust the `cost` argument of `cv.glm`. For more details, see the function documentation by running `?cv.glm`:

In [73]:
?cv.glm

cv.glm {boot},R Documentation
data,"A matrix or data frame containing the data. The rows should be cases and the columns correspond to variables, one of which is the response."
glmfit,"An object of class ""glm"" containing the results of a generalized linear model fitted to data."
cost,A function of two vector arguments specifying the cost function for the cross-validation. The first argument to cost should correspond to the observed responses and the second argument should correspond to the predicted or fitted responses from the generalized linear model. cost must return a non-negative scalar value. The default is the average squared error function.
K,The number of groups into which the data should be split to estimate the cross-validation prediction error. The value of K must be such that all groups are of approximately equal size. If the supplied value of K does not satisfy this criterion then it will be set to the closest integer which does and a warning is generated specifying the value of K used. The default is to set K equal to the number of observations in data which gives the usual leave-one-out cross-validation.
call,The original call to cv.glm.
K,The value of K used for the K-fold cross validation.
delta,A vector of length two. The first component is the raw cross-validation estimate of prediction error. The second component is the adjusted cross-validation estimate. The adjustment is designed to compensate for the bias introduced by not using leave-one-out cross-validation.
seed,The value of .Random.seed when cv.glm was called.


Here, we see `cost` is defined as:
> "A function of two vector arguments specifying the cost function for the cross-validation. The first argument to cost should correspond to the **observed responses** and the second argument should correspond to the **predicted or fitted responses** from the generalized linear model."

In the example code (scroll to bottom of the docs), we see that the appropriate cost function for a binary classification is

```
cost <- function(r, pi = 0) {
  mean(abs(r - pi) > 0.5)
}
```

Where `r` is the vector of observed responses (technically "pos" and "neg", but R treats these as 1 and 0 under the hood), and `pi` is the vector of *probabilities* (not log-odds) fit by the model. Thus, this boils down to our error: what proportion of observations were incorrectly classified. You will need to include this code below.

In [82]:
library(boot) # Load the bootstrap library

r <- results$actual
pi <- results$predicted
cost <- function(r, pi = 0) {
  mean(abs(r - pi) > 0.5)
}

model <- glm(diabetes ~ pregnant + glucose + pressure + triceps + mass + pedigree + age, 
             family = "binomial",
             data = cleaned_data) 

cv.err  = cv.glm(cleaned_data, model, cost=cost, K=nrow(cleaned_data))
cv.err$delta



[1] 0.2218045 0.2221154

How do your results compare to your manual LOOCV above?
> * They are the same. Yay!

---
## 4. Adjusting K and Reflection (2 points)

Recall that LOOCV has some drawbacks. In particular, it has quite high *variance* which can lead to poor performance on new test data. We can reduce this variance by increasing K.

Below, re-run your cross validation using `cv.glm` with `k` set to 3, 5, 10, and 15.

In [84]:
set.seed(1)
r <- results$actual
pi <- results$predicted
cost <- function(r, pi = 0) {
  mean(abs(r - pi) > 0.5)
}

model <- glm(diabetes ~ pregnant + glucose + pressure + triceps + mass + pedigree + age, 
             family = "binomial",
             data = cleaned_data) 

# K = 3
cv.err  = cv.glm(cleaned_data, model, cost=cost, K=3)
cv.err$delta

# K = 5
cv.err  = cv.glm(cleaned_data, model, cost=cost, K=5)
cv.err$delta

# K = 10
cv.err  = cv.glm(cleaned_data, model, cost=cost, K=10)
cv.err$delta

# K = 15
cv.err  = cv.glm(cleaned_data, model, cost=cost, K=15)
cv.err$delta

[1] 0.2105263 0.2168085

[1] 0.2161654 0.2184444

[1] 0.2236842 0.2230942

[1] 0.2274436 0.2281856

#### Reflection

How do your errors compare to your LOOCV error above? How do they change as k increases?
> K=3 and K=5
> *  

If you change the random seed above, you'll get slightly different errors. If you were to do the same with your LOOCV above , would you expect to get different results each time? Why or why not?
> * *Write response here*
> *

**DUE:** 11:59pm March 19, 2026

**IMPORTANT** Did you collaborate with anyone on this assignment? If so, list their names here.
> *Someone's Name*
>
>

**GenAI Utilization** Did you utilize any generative AI tools on this assignment? If so, please list the item and the paste respective prompt you used.

>
>